# Inteligência Empresarial - Análise de Dados de Vendas 📊

Este notebook contém a análise de dados das planilhas de vendas globais e gera gráficos para responder às questões propostas.

**Ferramentas utilizadas:** Python, Pandas, Matplotlib e Seaborn

## 1. Carregamento e Preparação dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

### 1.1 Carregar dados do Google Drive (Google Colab)

In [ ]:
# Para Google Colab - descomente as linhas abaixo
from google.colab import drive
drive.mount('/content/drive')

# Defina os caminhos dos arquivos
path_vendas = '/content/drive/My Drive/Vendas Globais.csv'
path_fornecedores = '/content/drive/My Drive/Fornecedores.csv'
path_transportadoras = '/content/drive/My Drive/Transportadoras.csv'
path_vendedores = '/content/drive/My Drive/Vendedores.csv'

# Alternativa: Carregar arquivos diretamente da URL (se estiverem no Drive com acesso público)
# Ou use os links de compartilhamento do Google Drive
# Substitua pelos IDs corretos dos arquivos no seu Google Drive

# Para execução local (descomente para testar localmente)
# path_vendas = 'Vendas Globais.csv'
# path_fornecedores = 'Fornecedores.csv'
# path_transportadoras = 'Transportadoras.csv'
# path_vendedores = 'Vendedores.csv'

### 1.2 Carregar os arquivos CSV

In [ ]:
# Descomente a opção apropriada:

# OPÇÃO 1: Google Drive (remova o # da primeira linha)
df_vendas = pd.read_csv(path_vendas)
df_fornecedores = pd.read_csv(path_fornecedores)
df_transportadoras = pd.read_csv(path_transportadoras)
df_vendedores = pd.read_csv(path_vendedores)

# OPÇÃO 2: Upload direto no Colab
from google.colab import files
uploaded = files.upload()
df_vendas = pd.read_csv('Vendas Globais.csv')
df_fornecedores = pd.read_csv('Fornecedores.csv')
df_transportadoras = pd.read_csv('Transportadoras.csv')
df_vendedores = pd.read_csv('Vendedores.csv')

# OPÇÃO 3: URLs do GitHub (substitua pelos URLs corretos seus)
# url_base = 'https://raw.githubusercontent.com/seu-usuario/seu-repo/main/sheets/'
# df_vendas = pd.read_csv(url_base + 'Vendas Globais.csv')
# df_fornecedores = pd.read_csv(url_base + 'Fornecedores.csv')
# df_transportadoras = pd.read_csv(url_base + 'Transportadoras.csv')
# df_vendedores = pd.read_csv(url_base + 'Vendedores.csv')

print('Dados carregados com sucesso!')
print(f'\nVendas Globais: {df_vendas.shape[0]} linhas x {df_vendas.shape[1]} colunas')
print(f'Fornecedores: {df_fornecedores.shape[0]} linhas x {df_fornecedores.shape[1]} colunas')
print(f'Transportadoras: {df_transportadoras.shape[0]} linhas x {df_transportadoras.shape[1]} colunas')
print(f'Vendedores: {df_vendedores.shape[0]} linhas x {df_vendedores.shape[1]} colunas')

### 1.3 Exploração inicial dos dados

In [ ]:
# Visualizar primeiras linhas
print('=== VENDAS GLOBAIS (Primeiras 5 linhas) ===')
print(df_vendas.head())
print('\n=== INFORMAÇÕES DO DATAFRAME VENDAS ===')
print(df_vendas.info())
print('\n=== ESTATÍSTICAS DESCRITIVAS ===')
print(df_vendas.describe())

### 1.4 Limpeza e transformação dos dados

In [ ]:
# Converter coluna de data para formato datetime
df_vendas['Data'] = pd.to_datetime(df_vendas['Data'], format='%d/%m/%Y')

# Extrair ano da data
df_vendas['Ano'] = df_vendas['Data'].dt.year
df_vendas['Mês'] = df_vendas['Data'].dt.month

# Verificar dados faltantes
print('=== DADOS FALTANTES ===')
print(df_vendas.isnull().sum())

# Remover linhas com valores faltantes críticos (se houver)
df_vendas = df_vendas.dropna(subset=['Vendas', 'ClienteID', 'ClientePaís'])

print(f'\nDataset após limpeza: {df_vendas.shape[0]} registros')
print(f'Anos disponíveis: {sorted(df_vendas["Ano"].unique())}')

## 2. PERGUNTA 1: Top 10 Maiores Clientes em Termos de Vendas

In [ ]:
# Análise: Quem são os 10 maiores clientes em termos de vendas ($)?

top_10_clientes = df_vendas.groupby('ClienteNome')['Vendas'].sum().sort_values(ascending=False).head(10)

print('=== TOP 10 MAIORES CLIENTES EM VENDAS ===')
print(top_10_clientes)
print(f'\nTotal de vendas (Top 10): ${top_10_clientes.sum():.2f}')

In [ ]:
# Gráfico 1: Top 10 Maiores Clientes
fig, ax = plt.subplots(figsize=(14, 8))

colors = plt.cm.viridis(np.linspace(0, 1, len(top_10_clientes)))
bars = ax.barh(range(len(top_10_clientes)), top_10_clientes.values, color=colors)

ax.set_yticks(range(len(top_10_clientes)))
ax.set_yticklabels(top_10_clientes.index, fontsize=11)
ax.set_xlabel('Vendas ($)', fontsize=12, fontweight='bold')
ax.set_title('Top 10 Maiores Clientes em Termos de Vendas ($)', fontsize=14, fontweight='bold', pad=20)
ax.invert_yaxis()

# Adicionar valores nas barras
for i, (bar, value) in enumerate(zip(bars, top_10_clientes.values)):
    ax.text(value, bar.get_y() + bar.get_height()/2, f'${value:,.0f}', 
            va='center', ha='left', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print('\n✅ Gráfico 1 gerado com sucesso!')

## 3. PERGUNTA 2: Três Maiores Países em Termos de Vendas

In [ ]:
# Análise: Quais os três maiores países em termos de vendas ($)?

top_3_paises = df_vendas.groupby('ClientePaís')['Vendas'].sum().sort_values(ascending=False).head(3)

print('=== TOP 3 MAIORES PAÍSES EM VENDAS ===')
print(top_3_paises)
print(f'\nPercentual do total de vendas:')
total_vendas = df_vendas['Vendas'].sum()
for pais, vendas in top_3_paises.items():
    percentual = (vendas / total_vendas) * 100
    print(f'{pais}: ${vendas:,.2f} ({percentual:.2f}%)')

In [ ]:
# Gráfico 2: Top 3 Maiores Países
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico de barras
colors_paises = ['#FF6B6B', '#4ECDC4', '#45B7D1']
bars = ax1.bar(top_3_paises.index, top_3_paises.values, color=colors_paises, edgecolor='black', linewidth=2)
ax1.set_ylabel('Vendas ($)', fontsize=12, fontweight='bold')
ax1.set_title('Top 3 Maiores Países em Vendas ($)', fontsize=13, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Adicionar valores nas barras
for bar, value in zip(bars, top_3_paises.values):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'${value:,.0f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# Gráfico de pizza
explode = (0.05, 0.05, 0.05)
wedges, texts, autotexts = ax2.pie(top_3_paises.values, labels=top_3_paises.index, autopct='%1.1f%%',
                                     colors=colors_paises, explode=explode, startangle=90,
                                     textprops={'fontsize': 11, 'fontweight': 'bold'})
ax2.set_title('Distribuição de Vendas - Top 3 Países', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print('\n✅ Gráfico 2 gerado com sucesso!')

## 4. PERGUNTA 3: Categorias de Produtos com Maior Faturamento no Brasil

In [ ]:
# Análise: Quais as categorias de produtos que geram maior faturamento no Brasil?

brasil = df_vendas[df_vendas['ClientePaís'] == 'Brazil'].copy()

if len(brasil) > 0:
    categorias_brasil = brasil.groupby('CategoriaNome')['Vendas'].sum().sort_values(ascending=False)
    
    print('=== CATEGORIAS COM MAIOR FATURAMENTO NO BRASIL ===')
    print(categorias_brasil)
    print(f'\nTotal de vendas no Brasil: ${brasil["Vendas"].sum():,.2f}')
else:
    print('Nenhum registro para Brazil encontrado. Verificando nomes de países...')
    print(df_vendas['ClientePaís'].unique())

In [ ]:
# Gráfico 3: Categorias com Maior Faturamento no Brasil
if len(brasil) > 0:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    colors = plt.cm.Spectral(np.linspace(0, 1, len(categorias_brasil)))
    bars = ax.barh(range(len(categorias_brasil)), categorias_brasil.values, color=colors, edgecolor='black', linewidth=1.5)
    
    ax.set_yticks(range(len(categorias_brasil)))
    ax.set_yticklabels(categorias_brasil.index, fontsize=11)
    ax.set_xlabel('Faturamento ($)', fontsize=12, fontweight='bold')
    ax.set_title('Categorias de Produtos com Maior Faturamento no Brasil', fontsize=14, fontweight='bold', pad=20)
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)
    
    # Adicionar valores nas barras
    for bar, value in zip(bars, categorias_brasil.values):
        ax.text(value, bar.get_y() + bar.get_height()/2, f'${value:,.0f}',
                va='center', ha='left', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print('\n✅ Gráfico 3 gerado com sucesso!')
else:
    print('⚠️ Nenhum dado disponível para Brasil')

## 5. PERGUNTA 4: Despesa com Frete por Transportadora

In [ ]:
# Análise: Qual a despesa com frete envolvendo cada transportadora?

# Merge com tabela de transportadoras
df_vendas_transportadora = df_vendas.merge(df_transportadoras, on='TransportadoraID', how='left')

despesa_frete = df_vendas_transportadora.groupby('TransportadoraNome')['Frete'].sum().sort_values(ascending=False)

print('=== DESPESA COM FRETE POR TRANSPORTADORA ===')
print(despesa_frete)
print(f'\nTotal gasto em frete: ${despesa_frete.sum():,.2f}')
print(f'\nPercentual por transportadora:')
for transportadora, valor in despesa_frete.items():
    percentual = (valor / despesa_frete.sum()) * 100
    print(f'{transportadora}: ${valor:,.2f} ({percentual:.2f}%)')

In [ ]:
# Gráfico 4: Despesa com Frete por Transportadora
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico de barras
colors_frete = plt.cm.Set3(np.linspace(0, 1, len(despesa_frete)))
bars = ax1.bar(despesa_frete.index, despesa_frete.values, color=colors_frete, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('Despesa com Frete ($)', fontsize=12, fontweight='bold')
ax1.set_title('Despesa com Frete por Transportadora', fontsize=13, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=15, ha='right')

# Adicionar valores nas barras
for bar, value in zip(bars, despesa_frete.values):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'${value:,.0f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# Gráfico de pizza
explode = tuple([0.05 if i == 0 else 0.02 for i in range(len(despesa_frete))])
wedges, texts, autotexts = ax2.pie(despesa_frete.values, labels=despesa_frete.index, autopct='%1.1f%%',
                                     colors=colors_frete, explode=explode, startangle=45,
                                     textprops={'fontsize': 11, 'fontweight': 'bold'})
ax2.set_title('Distribuição de Frete por Transportadora', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print('\n✅ Gráfico 4 gerado com sucesso!')

## 6. PERGUNTA 5: Principais Clientes de Calçados Masculinos na Alemanha

In [ ]:
# Análise: Quais são os principais clientes do segmento "Men´s Footwear" na Alemanha?

print('=== CATEGORIAS DISPONÍVEIS ===')
print(df_vendas['CategoriaNome'].unique())

# Filtrar especificamente para "Men´s Footwear" (nota: apóstrofo especial)
mensfw = df_vendas[df_vendas['CategoriaNome'] == "Men´s Footwear"].copy()
print(f'\nTotal de registros de Men´s Footwear: {len(mensfw)}')

# Filtrar para Alemanha (verificar nomes exatos)
mensfw_germany = mensfw[mensfw['ClientePaís'] == 'Germany'].copy()

if len(mensfw_germany) > 0:
    principais_clientes_calcados = mensfw_germany.groupby('ClienteNome')['Vendas'].sum().sort_values(ascending=False).head(10)
    
    print('\n=== PRINCIPAIS CLIENTES DE CALÇADOS MASCULINOS (MEN´S FOOTWEAR) NA ALEMANHA ===')
    print(principais_clientes_calcados)
    print(f'\nTotal de vendas: ${principais_clientes_calcados.sum():,.2f}')
else:
    print('\n⚠️ Nenhum registro encontrado com filtros atuais.')
    print('Verificando dados disponíveis...')
    print(f'\nPaíses com Men´s Footwear: {mensfw["ClientePaís"].unique()}')

In [ ]:
# Gráfico 5: Principais Clientes de Calçados Masculinos na Alemanha
if len(mensfw_germany) > 0 and len(principais_clientes_calcados) > 0:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(principais_clientes_calcados)))
    bars = ax.barh(range(len(principais_clientes_calcados)), principais_clientes_calcados.values, color=colors, edgecolor='black', linewidth=1.5)
    
    ax.set_yticks(range(len(principais_clientes_calcados)))
    ax.set_yticklabels(principais_clientes_calcados.index, fontsize=11)
    ax.set_xlabel('Vendas ($)', fontsize=12, fontweight='bold')
    ax.set_title('Principais Clientes de Calçados Masculinos (Men´s Footwear) na Alemanha', fontsize=14, fontweight='bold', pad=20)
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)
    
    # Adicionar valores nas barras
    for bar, value in zip(bars, principais_clientes_calcados.values):
        ax.text(value, bar.get_y() + bar.get_height()/2, f'${value:,.0f}',
                va='center', ha='left', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print('\n✅ Gráfico 5 gerado com sucesso!')
else:
    print('⚠️ Nenhum dado disponível para gerar o gráfico')

## 7. PERGUNTA 6: Vendedores que Mais Dão Descontos nos EUA

In [ ]:
# Análise: Quais os vendedores que mais dão descontos nos EUA?

df_vendas_vendedor = df_vendas.merge(df_vendedores, on='VendedorID', how='left')

eua = df_vendas_vendedor[df_vendas_vendedor['ClientePaís'].str.contains('USA|United States|Estados Unidos', case=False, na=False)].copy()

if len(eua) > 0:
    vendedores_desconto_eua = eua.groupby('VendedorNome')['Desconto'].sum().sort_values(ascending=False).head(10)
    
    print('=== VENDEDORES COM MAIOR DESCONTO NOS EUA ===')
    print(vendedores_desconto_eua)
    print(f'\nTotal de descontos (EUA): ${eua["Desconto"].sum():,.2f}')
else:
    print('Verificando nomes de países nos dados...')
    print(df_vendas_vendedor['ClientePaís'].unique()[:15])

In [ ]:
# Gráfico 6: Vendedores que Mais Dão Descontos nos EUA
if len(eua) > 0 and len(vendedores_desconto_eua) > 0:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    colors = plt.cm.Purples(np.linspace(0.4, 0.9, len(vendedores_desconto_eua)))
    bars = ax.barh(range(len(vendedores_desconto_eua)), vendedores_desconto_eua.values, color=colors, edgecolor='black', linewidth=1.5)
    
    ax.set_yticks(range(len(vendedores_desconto_eua)))
    ax.set_yticklabels(vendedores_desconto_eua.index, fontsize=11)
    ax.set_xlabel('Desconto Total ($)', fontsize=12, fontweight='bold')
    ax.set_title('Vendedores que Mais Dão Descontos nos EUA', fontsize=14, fontweight='bold', pad=20)
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)
    
    # Adicionar valores nas barras
    for bar, value in zip(bars, vendedores_desconto_eua.values):
        ax.text(value, bar.get_y() + bar.get_height()/2, f'${value:,.0f}',
                va='center', ha='left', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print('\n✅ Gráfico 6 gerado com sucesso!')
else:
    print('⚠️ Nenhum dado disponível para esta análise')

## 8. PERGUNTA 7: Fornecedores com Maior Margem de Lucro em Vestuário Feminino

In [ ]:
# Análise: Quais os fornecedores que dão a maior margem de lucro no segmento "Womens wear"?

df_vendas_fornecedor = df_vendas.merge(df_fornecedores, on='FornecedorID', how='left')

vestuario_feminino = df_vendas_fornecedor[df_vendas_fornecedor['CategoriaNome'].str.contains('Womens|Feminino', case=False, na=False)].copy()

if len(vestuario_feminino) > 0:
    # 'Margem Bruta' representa a margem de lucro
    fornecedores_margem = vestuario_feminino.groupby('FornecedorNome')['Margem Bruta'].sum().sort_values(ascending=False).head(10)
    
    print('=== FORNECEDORES COM MAIOR MARGEM DE LUCRO EM VESTUÁRIO FEMININO ===')
    print(fornecedores_margem)
    print(f'\nTotal de margem (Vestuário Feminino): ${vestuario_feminino["Margem Bruta"].sum():,.2f}')
else:
    print('Nenhum registro encontrado para Vestuário Feminino')
    print('Categorias disponíveis:', df_vendas_fornecedor['CategoriaNome'].unique())

In [ ]:
# Gráfico 7: Fornecedores com Maior Margem de Lucro em Vestuário Feminino
if len(vestuario_feminino) > 0 and len(fornecedores_margem) > 0:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    colors = plt.cm.Reds(np.linspace(0.4, 0.9, len(fornecedores_margem)))
    bars = ax.barh(range(len(fornecedores_margem)), fornecedores_margem.values, color=colors, edgecolor='black', linewidth=1.5)
    
    ax.set_yticks(range(len(fornecedores_margem)))
    ax.set_yticklabels(fornecedores_margem.index, fontsize=11)
    ax.set_xlabel('Margem Bruta Total ($)', fontsize=12, fontweight='bold')
    ax.set_title('Fornecedores com Maior Margem de Lucro em Vestuário Feminino (Womens wear)', fontsize=14, fontweight='bold', pad=20)
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)
    
    # Adicionar valores nas barras
    for bar, value in zip(bars, fornecedores_margem.values):
        ax.text(value, bar.get_y() + bar.get_height()/2, f'${value:,.0f}',
                va='center', ha='left', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print('\n✅ Gráfico 7 gerado com sucesso!')
else:
    print('⚠️ Nenhum dado disponível para esta análise')

## 9. PERGUNTA 8: Vendas Totais em 2009 e Evolução 2009-2012

In [ ]:
# Análise: Quanto foi vendido em 2009? E evolução anual 2009-2012

vendas_2009 = df_vendas[df_vendas['Ano'] == 2009]['Vendas'].sum()

print('=== VENDAS TOTAIS EM 2009 ===')
print(f'Total: ${vendas_2009:,.2f}')

# Evolução anual 2009-2012
evolucao_anual = df_vendas[(df_vendas['Ano'] >= 2009) & (df_vendas['Ano'] <= 2012)].groupby('Ano')['Vendas'].sum().sort_index()

print('\n=== EVOLUÇÃO ANUAL (2009-2012) ===')
for ano, vendas in evolucao_anual.items():
    print(f'{ano}: ${vendas:,.2f}')

In [ ]:
# Gráfico 8.1: Vendas Totais em 2009
fig, ax = plt.subplots(figsize=(10, 6))

# Criar um gráfico de gauge simples usando barra horizontal
ax.barh(['2009'], [vendas_2009], color='#2ECC71', height=0.5, edgecolor='black', linewidth=2)
ax.set_xlim(0, vendas_2009 * 1.1)
ax.set_ylabel('Ano', fontsize=12, fontweight='bold')
ax.set_xlabel('Vendas Totais ($)', fontsize=12, fontweight='bold')
ax.set_title('Vendas Totais em 2009', fontsize=14, fontweight='bold', pad=20)

# Adicionar valor
ax.text(vendas_2009/2, 0, f'${vendas_2009:,.0f}', 
        ha='center', va='center', fontsize=14, fontweight='bold', color='white')

plt.tight_layout()
plt.show()

print('\n✅ Gráfico 8.1 gerado com sucesso!')

In [ ]:
# Gráfico 8.2: Evolução Anual 2009-2012
fig, ax = plt.subplots(figsize=(14, 7))

# Gráfico de linha com área
anos = evolucao_anual.index
vendas = evolucao_anual.values

ax.plot(anos, vendas, marker='o', linewidth=3, markersize=12, color='#3498DB', label='Vendas')
ax.fill_between(anos, vendas, alpha=0.3, color='#3498DB')

ax.set_xlabel('Ano', fontsize=12, fontweight='bold')
ax.set_ylabel('Vendas ($)', fontsize=12, fontweight='bold')
ax.set_title('Evolução de Vendas Anuais (2009-2012)', fontsize=14, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3)
ax.set_xticks(anos)

# Adicionar valores nos pontos
for ano, venda in zip(anos, vendas):
    ax.text(ano, venda, f'${venda:,.0f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Calcular variação percentual
variacao = ((vendas_2009 - vendas_2012) / vendas_2012 * 100) if len(vendas) > 0 else 0
vendas_2012 = evolucao_anual.iloc[-1] if len(evolucao_anual) > 0 else 0
variacao = ((vendas_2009 - vendas_2012) / vendas_2012 * 100) if vendas_2012 != 0 else 0

plt.tight_layout()
plt.show()

print('\n✅ Gráfico 8.2 gerado com sucesso!')
print(f'\nVariação de 2009 para 2012: {variacao:.2f}%')

## 10. PERGUNTA 9: Principais Clientes de Calçados Masculinos em 2013

In [ ]:
# Análise: Principais clientes de Calçados Masculinos em 2013

dados_2013 = df_vendas[df_vendas['Ano'] == 2013].copy()

if len(dados_2013) > 0:
    calcados_2013 = dados_2013[dados_2013['CategoriaNome'].str.contains('Footwear', case=False, na=False)].copy()
    
    if len(calcados_2013) > 0:
        print('=== PRINCIPAIS CLIENTES DE CALÇADOS MASCULINOS EM 2013 ===')
        print('Por cidades:')
        clientes_2013 = calcados_2013.groupby(['ClienteNome', 'ClienteCidade']).agg({'Vendas': 'sum'}).sort_values('Vendas', ascending=False).head(10)
        print(clientes_2013)
    else:
        print('✅ Nenhum registro de Calçados Masculinos em 2013 (conforme esperado no README)')
else:
    print('✅ Nenhum registro em 2013 (conforme esperado no README)')

In [ ]:
# Nota sobre a Pergunta 9
print('='*70)
print('PERGUNTA 9: Quais são os principais clientes de Calçados Masculinos em 2013?')
print('='*70)
print('\n📌 RESULTADO: Não há registros em 2013')
print('\nConfirme verificando os anos disponíveis nos dados:')
print(sorted(df_vendas['Ano'].unique()))

## 11. PERGUNTA 10: Vendas na Europa por País

In [ ]:
# Análise: Na Europa, quanto se vende para cada país?

# Lista de países europeus
paises_europa = ['France', 'Germany', 'UK', 'United Kingdom', 'Italy', 'Spain', 'Portugal', 
                 'Netherlands', 'Belgium', 'Greece', 'Poland', 'Sweden', 'Norway', 'Denmark',
                 'Finland', 'Ireland', 'Austria', 'Switzerland', 'Czech', 'Hungary', 'Romania',
                 'Bulgaria', 'Croatia', 'Slovenia', 'Slovakia', 'Lithuania', 'Latvia', 'Estonia',
                 'FRA', 'DEU', 'GBR', 'ITA', 'ESP', 'PRT', 'NLD', 'BEL', 'GRC', 'POL', 'SWE', 'NOR', 'DNK']

# Filtrar dados para Europa
europa = df_vendas[df_vendas['ClientePaís'].isin(paises_europa) | 
                    df_vendas['ClientePaís'].str.contains('France|Germany|UK|Italy|Spain|Portugal|Netherlands|Belgium', case=False, na=False)].copy()

if len(europa) > 0:
    vendas_europa = europa.groupby('ClientePaís')['Vendas'].sum().sort_values(ascending=False)
    
    print('=== VENDAS NA EUROPA POR PAÍS ===')
    print(vendas_europa)
    print(f'\nTotal de vendas na Europa: ${vendas_europa.sum():,.2f}')
else:
    print('Verificando nomes de países europeus nos dados...')
    print(df_vendas['ClientePaís'].unique())

In [ ]:
# Gráfico 10: Vendas na Europa por País
if len(europa) > 0:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    colors = plt.cm.tab20(np.linspace(0, 1, len(vendas_europa)))
    bars = ax.barh(range(len(vendas_europa)), vendas_europa.values, color=colors, edgecolor='black', linewidth=1)
    
    ax.set_yticks(range(len(vendas_europa)))
    ax.set_yticklabels(vendas_europa.index, fontsize=11)
    ax.set_xlabel('Vendas ($)', fontsize=12, fontweight='bold')
    ax.set_title('Vendas na Europa por País', fontsize=14, fontweight='bold', pad=20)
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)
    
    # Adicionar valores nas barras
    for bar, value in zip(bars, vendas_europa.values):
        ax.text(value, bar.get_y() + bar.get_height()/2, f'${value:,.0f}',
                va='center', ha='left', fontsize=9, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print('\n✅ Gráfico 10 gerado com sucesso!')
else:
    print('⚠️ Nenhum dado disponível para esta análise')

## 12. Resumo da Análise

In [ ]:
print('='*70)
print('RESUMO EXECUTIVO - ANÁLISE DE VENDAS GLOBAIS')
print('='*70)

print(f'\n📊 ESTATÍSTICAS GERAIS:')
print(f'   Total de Registros: {len(df_vendas):,}')
print(f'   Total de Vendas: ${df_vendas["Vendas"].sum():,.2f}')
print(f'   Total de Descontos: ${df_vendas["Desconto"].sum():,.2f}')
print(f'   Total de Frete: ${df_vendas["Frete"].sum():,.2f}')
print(f'   Margem Bruta Total: ${df_vendas["Margem Bruta"].sum():,.2f}')
print(f'   Período: {df_vendas["Data"].min().strftime("%d/%m/%Y")} a {df_vendas["Data"].max().strftime("%d/%m/%Y")}')

print(f'\n🌍 COBERTURA GEOGRÁFICA:')
print(f'   Total de Países: {df_vendas["ClientePaís"].nunique()}')
print(f'   Total de Cidades: {df_vendas["ClienteCidade"].nunique()}')

print(f'\n👥 ATORES:')
print(f'   Total de Clientes: {df_vendas["ClienteID"].nunique()}')
print(f'   Total de Vendedores: {df_vendas["VendedorID"].nunique()}')
print(f'   Total de Fornecedores: {df_vendas["FornecedorID"].nunique()}')
print(f'   Total de Transportadoras: {df_vendas["TransportadoraID"].nunique()}')

print(f'\n📦 PRODUTOS E CATEGORIAS:')
print(f'   Total de Produtos: {df_vendas["ProdutoID"].nunique()}')
print(f'   Total de Categorias: {df_vendas["CategoriaID"].nunique()}')

print(f'\n✅ ANÁLISES REALIZADAS:')
print('   ✓ Top 10 Maiores Clientes em Vendas')
print('   ✓ Top 3 Maiores Países em Vendas')
print('   ✓ Categorias com Maior Faturamento no Brasil')
print('   ✓ Despesa com Frete por Transportadora')
print('   ✓ Principais Clientes de Calçados Masculinos na Alemanha')
print('   ✓ Vendedores que Mais Dão Descontos nos EUA')
print('   ✓ Fornecedores com Maior Margem em Vestuário Feminino')
print('   ✓ Vendas Totais em 2009 e Evolução 2009-2012')
print('   ✓ Principais Clientes de Calçados em 2013 (Sem registros)')
print('   ✓ Vendas na Europa por País')

print('\n' + '='*70)
print('📈 Análise concluída com sucesso!')
print('='*70)